In [22]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid", context="talk")

ancovas_mean_results_path = '../../../../results/pet/paper_analysis/ancova_rois_mean/'
ancovas_cerebellum_results_path = '../../../../results/pet/paper_analysis/ancova_rois_cerebellum/'
neurocloud_results_path = '../../../../results/pet/paper_analysis/neurocloud/'
rois_to_lobe_csv = os.path.join('../../../../data/pet/', 'hammers_roi_to_lobe.csv')
rois_abbreviations_csv = os.path.join('../../../../data/pet/', 'roi_abbreviations.csv')
output_path = os.path.join('../../../../results/pet/paper_analysis/', 'consensus/')
os.makedirs(output_path, exist_ok=True)

# Load rois to lobe mapping
rois_to_lobe_df = pd.read_csv(rois_to_lobe_csv)
# Load abbreviations for ROIs
rois_abbreviations_df = pd.read_csv(rois_abbreviations_csv)


In [23]:
# Perform consensus analysis between ceunim controls, cermep and neurocloud
csv_ceunim_controls_filename = os.path.join(ancovas_mean_results_path, 'ancova_lc_vs_ceunim_controls_results_time_to_acquisition.csv')
csv_cermep_filename = os.path.join(ancovas_mean_results_path, 'ancova_lc_vs_cermep_controls_results.csv')
csv_neurocloud_filename = os.path.join(neurocloud_results_path, 'ttest_neurocloud_long_covid_hammers_names.csv')  
# Read each csv
ceunim_controls_df = pd.read_csv(csv_ceunim_controls_filename)
cermep_df = pd.read_csv(csv_cermep_filename)
neurocloud_df = pd.read_csv(csv_neurocloud_filename)

# Now we identify:
# First, the ROIs that p_FDR<=0.05 for the three datasets
# For cerunim and cermep the fiels are: ROI,n,F_group,p_group,beta_group,mean_control,mean_lc,delta_lc_minus_control,direction,p_sex,beta_sex,p_age,beta_age,p_fdr,significant_fdr
# For neurocloud the fields are:Region,N,MEAN_Z,MEDIAN_Z,STD_Z,COHEN_D,TSTAT,PVAL_T,WSTAT,PVAL_W,Effect, P_FDR
ceunim_controls_significant_rois = ceunim_controls_df[ceunim_controls_df['p_fdr'] <= 0.05]['ROI']
cermep_significant_rois = cermep_df[cermep_df['p_fdr'] <= 0.05]['ROI']
neurocloud_significant_rois = neurocloud_df[neurocloud_df['PVAL_FDR'] <= 0.05]['Region']
print(ceunim_controls_significant_rois)
print(cermep_significant_rois)
print(neurocloud_significant_rois)
consensus_rois = set(ceunim_controls_significant_rois) & set(cermep_significant_rois) & set(neurocloud_significant_rois)
# and a second condition is that the direction is the same across the three datasets, therefore mean_lc and MEAN_Z have the same sign
consensus_rois_with_same_direction = set()
for roi in consensus_rois:
    mean_lc_ceunim = ceunim_controls_df.loc[ceunim_controls_df['ROI'] == roi, 'mean_lc'].values[0]
    mean_lc_cermep = cermep_df.loc[cermep_df['ROI'] == roi, 'mean_lc'].values[0]
    mean_z_neurocloud = neurocloud_df.loc[neurocloud_df['Region'] == roi, 'MEAN_Z'].values[0]
    if (mean_lc_ceunim > 0 and mean_lc_cermep > 0 and mean_z_neurocloud > 0) or (mean_lc_ceunim < 0 and mean_lc_cermep < 0 and mean_z_neurocloud < 0):
        consensus_rois_with_same_direction.add(roi)
print("Consensus ROIs with the same direction across all three datasets:")
print(consensus_rois_with_same_direction)
print("Consensus ROIs with p_FDR <= 0.05 for all three datasets:")
print(consensus_rois)

# Now add to the list those which are significant in at least two datasets
from collections import Counter
all_significant_rois = list(ceunim_controls_significant_rois) + list(cermep_significant_rois) + list(neurocloud_significant_rois)
roi_counts = Counter(all_significant_rois)
at_least_two_datasets_rois = {roi for roi, count in roi_counts.items() if count >= 2}
print("ROIs significant in at least two datasets:")
print(at_least_two_datasets_rois)

# Create a df with the consensus ROIs, the p_values, p_FDR and effect sizes and their directions
consensus_data = []
for roi in consensus_rois:
    ceunim_row = ceunim_controls_df[ceunim_controls_df['ROI'] == roi].iloc[0]
    cermep_row = cermep_df[cermep_df['ROI'] == roi].iloc[0]
    neurocloud_row = neurocloud_df[neurocloud_df['Region'] == roi].iloc[0]
    consensus_data.append({
        'ROI': roi,
        'ceunim_p': ceunim_row['p_group'],
        'ceunim_p_fdr': ceunim_row['p_fdr'],
        'ceunim_mean_lc': ceunim_row['mean_lc'],
        'cermep_p': cermep_row['p_group'],
        'cermep_p_fdr': cermep_row['p_fdr'],
        'cermep_mean_lc': cermep_row['mean_lc'],
        'neurocloud_p': neurocloud_row['PVAL_T'],
        'neurocloud_p_fdr': neurocloud_row['PVAL_FDR'],
        'neurocloud_mean_z': neurocloud_row['MEAN_Z'],
        'ceunim_effect': ceunim_row['delta_lc_minus_control'],
        'ceunim_direction': 'positive' if ceunim_row['delta_lc_minus_control'] > 0 else 'negative',
        'cermep_effect': cermep_row['delta_lc_minus_control'],
        'cermep_direction': 'positive' if cermep_row['delta_lc_minus_control'] > 0 else 'negative',
        'neurocloud_effect': neurocloud_row['MEAN_Z'],
        'neurocloud_direction': 'positive' if neurocloud_row['MEAN_Z'] > 0 else 'negative'
    })
consensus_df = pd.DataFrame(consensus_data)
print("Consensus ROIs with detailed statistics:")
print(consensus_df)

# Create a df with the ROIs significant in at least two datasets
at_least_two_datasets_data = []
for roi in at_least_two_datasets_rois:
    ceunim_row = ceunim_controls_df[ceunim_controls_df['ROI'] == roi].iloc[0]
    cermep_row = cermep_df[cermep_df['ROI'] == roi].iloc[0]
    # neurocloud may be missiing a roin,s o check that exists first, if not leave empty
    if roi in neurocloud_df['Region'].values:
        neurocloud_row = neurocloud_df[neurocloud_df['Region'] == roi].iloc[0]
        neurocloud_direction = 'positive' if neurocloud_row['MEAN_Z'] > 0 else 'negative'
    else:
        neurocloud_row = pd.Series({'PVAL_T': None, 'MEAN_Z': None, 'PVAL_FDR': None, 'neurocloud_direction': None})
    at_least_two_datasets_data.append({
         'ROI': roi,
                'ceunim_p': ceunim_row['p_group'],
                'ceunim_p_fdr': ceunim_row['p_fdr'],
                'ceunim_mean_lc': ceunim_row['mean_lc'],
                'cermep_p': cermep_row['p_group'],
                'cermep_p_fdr': cermep_row['p_fdr'],
                'cermep_mean_lc': cermep_row['mean_lc'],
                'neurocloud_p': neurocloud_row['PVAL_T'],
                'neurocloud_p_fdr': neurocloud_row['PVAL_FDR'],
                'neurocloud_mean_z': neurocloud_row['MEAN_Z'],
                'ceunim_effect': ceunim_row['delta_lc_minus_control'],
                'ceunim_direction': 'positive' if ceunim_row['delta_lc_minus_control'] > 0 else 'negative',
                'cermep_effect': cermep_row['delta_lc_minus_control'],
                'cermep_direction': 'positive' if cermep_row['delta_lc_minus_control'] > 0 else 'negative',
                'neurocloud_effect': neurocloud_row['MEAN_Z'],
                'neurocloud_direction': neurocloud_direction
    })
at_least_two_datasets_df = pd.DataFrame(at_least_two_datasets_data)
print("ROIs significant in at least two datasets with detailed statistics:")
print(at_least_two_datasets_df)

# Save them in csv files
consensus_df.to_csv(os.path.join(output_path, "consensus_rois.csv"), index=False)
at_least_two_datasets_df.to_csv(os.path.join(output_path, "at_least_two_datasets_rois.csv"), index=False)


0      TL-middle-and-inferior-temporal-gyrus-L
1                     PL-supramarginal-gyrus-L
2                   FL-lateral-orbital-gyrus-L
3                 TL-posterior-temporal-lobe-L
4                  FL-inferior-frontal-gyrus-L
5                    FL-middle-frontal-gyrus-L
6                CG-anterior-cingulate-gyrus-R
7                  FL-superior-frontal-gyrus-R
8            FL-pre-subgenual-frontal-cortex-R
9                 FL-posterior-orbital-gyrus-L
10     TL-middle-and-inferior-temporal-gyrus-R
11                 FL-superior-frontal-gyrus-L
12    TL-anterior-temporal-lobe-lateral-part-L
13                                cerebellum-R
14                          PL-angular-gyrus-L
15                insula-anterior-long-gyrus-R
16                                  pallidum-L
17               insula-anterior-short-gyrus-R
18                    PL-supramarginal-gyrus-R
19              CG-posterior-cingulate-gyrus-L
20               FL-subgenual-frontal-cortex-R
21           

PermissionError: [Errno 13] Permission denied: '../../../../results/pet/paper_analysis/consensus/consensus_rois.csv'